In [1]:
# %%
from pathlib import Path
from collections import Counter

import copy
import numpy as np
import pandas as pd
import nibabel as nib
import torch
import torch.nn as nn

from nibabel.processing import resample_to_output
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score, f1_score
from torch.utils.data import Dataset, DataLoader

from resources.cut_off_edges import cut_off_edges
from resources.standarize_image import standardize_image

from notebooks.rotate_nifti_by_errors import rotate_nifti_by_errors
from notebooks.get_rotation_errors_from_ants import get_rotation_errors_from_ants

In [2]:
# %%
project_dir = Path.cwd().parents[1]

if not (project_dir / "data").exists():
    project_dir = project_dir.parent

niftis_dir = project_dir / "data" / "niftis"
labels_path = niftis_dir / "train_labels_JNDlMjr.csv"

outputs_dir = project_dir / "src" / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

labels_df = pd.read_csv(labels_path)
labels = labels_df.set_index("uid")["is_pathologic"]

nifti_paths = sorted(niftis_dir.glob("*.nii.gz"))

print("Katalog projektu:", project_dir)
print("Liczba obrazów:", len(nifti_paths))
print("Liczba etykiet:", len(labels))

Katalog projektu: C:\Users\abram\PycharmProjects\DrivenData_DAT_Parkinsons
Liczba obrazów: 1363
Liczba etykiet: 1362


In [3]:
# %%
all_uids = []
all_paths = []
all_y = []

for nifti_path in nifti_paths:
    uid = nifti_path.name.removesuffix(".nii.gz")

    if uid not in labels.index:
        print(f"Brak etykiety dla {uid}")
        continue

    all_uids.append(uid)
    all_paths.append(nifti_path)
    all_y.append(int(labels.loc[uid]))

all_uids = np.asarray(all_uids)
all_paths = np.asarray(all_paths)
all_y = np.asarray(all_y, dtype=np.int64)

print("UID:", all_uids.shape)
print("y:", np.unique(all_y, return_counts=True))

Brak etykiety dla __01nouhtc
UID: (1362,)
y: (array([0, 1]), array([615, 747]))


In [4]:
# %%
indices = np.arange(len(all_y))

train_val_indices, test_indices = train_test_split(
    indices,
    test_size=0.2,
    stratify=all_y,
    random_state=42,
)

train_val_paths = all_paths[train_val_indices]
train_val_uids = all_uids[train_val_indices]
train_val_y = all_y[train_val_indices]

test_paths = all_paths[test_indices]
test_uids = all_uids[test_indices]
test_y = all_y[test_indices]

print("Train_val:", len(train_val_y), np.unique(train_val_y, return_counts=True))
print("Test:", len(test_y), np.unique(test_y, return_counts=True))

Train_val: 1089 (array([0, 1]), array([492, 597]))
Test: 273 (array([0, 1]), array([123, 150]))


In [19]:
# %%
def preprocess_one_nifti(nifti_path, random_seed=None):
    rotation_error, tilt_error, pitch_error = get_rotation_errors_from_ants(
        nifti_path,
        random_seed=random_seed,
    )

    image = nib.load(nifti_path)

    image = rotate_nifti_by_errors(
        image,
        rotation_error=rotation_error,
        tilt_error=tilt_error,
        pitch_error=pitch_error,
        order=1,
    )

    processed_image = cut_off_edges(image)

    processed_image = cut_off_edges(
        processed_image,
        x_left=35,
        x_right=35,
        y_front=40,
        y_back=40,
        z_down=30,
        z_up=30,
    )

    processed_image = standardize_image(
        processed_image,
        percent_top=0.02,
    )

    processed_image = resample_to_output(
        processed_image,
        voxel_sizes=(2.46, 2.46, 2.46),
        order=1,
    )

    processed_image = cut_off_edges(
        processed_image,
        x_left=30,
        x_right=30,
        y_front=30,
        y_back=30,
        z_down=20,
        z_up=25,
    )

    processed_image = cut_off_edges(
        processed_image,
        x_left=25,
        x_right=25,
        y_front=30,
        y_back=10,
        z_down=10,
        z_up=25,
    )

    volume = processed_image.get_fdata(dtype=np.float32)

    return volume

In [20]:
# %%
from collections import Counter
from pathlib import Path

import numpy as np


def build_augmented_dataset(
    paths,
    y_values,
    uids,
    augment_count,
    base_seed,
    checkpoint_dir,
):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    volumes = []
    y_out = []
    uids_out = []
    aug_ids = []

    dataset_size = len(paths)

    for index, (nifti_path, y_value, uid) in enumerate(
        zip(paths, y_values, uids),
        start=1,
    ):
        for aug_id in range(augment_count):

            # Inny, ale zawsze powtarzalny seed dla każdego obrazu
            # i każdej augmentacji.
            random_seed = (
                base_seed
                + (index - 1) * augment_count
                + aug_id
            )

            checkpoint_path = (
                checkpoint_dir
                / f"{uid}_aug_{aug_id}_seed_{random_seed}.npy"
            )

            if checkpoint_path.exists():
                print(
                    f"{index}/{dataset_size} | {uid} | "
                    f"aug {aug_id + 1}/{augment_count} | wczytano",
                    flush=True,
                )

                volume = np.load(
                    checkpoint_path,
                    allow_pickle=False,
                )

            else:
                print(
                    f"{index}/{dataset_size} | {uid} | "
                    f"aug {aug_id + 1}/{augment_count} | przetwarzanie",
                    flush=True,
                )

                volume = preprocess_one_nifti(
                    nifti_path,
                    random_seed=random_seed,
                ).astype(np.float32)

                # Najpierw zapis do pliku tymczasowego.
                # Chroni to przed pozostawieniem uszkodzonego checkpointu,
                # jeżeli komputer wyłączy się podczas zapisywania.
                temporary_path = (
                    checkpoint_dir
                    / f"{uid}_aug_{aug_id}_seed_{random_seed}.tmp.npy"
                )

                np.save(
                    temporary_path,
                    volume,
                    allow_pickle=False,
                )

                temporary_path.replace(checkpoint_path)

            volumes.append(volume)
            y_out.append(y_value)
            uids_out.append(uid)
            aug_ids.append(aug_id)

    shape_counts = Counter(
        volume.shape
        for volume in volumes
    )
    print("Shape counts:", shape_counts)

    X = np.stack(volumes).astype(np.float32)
    X = X[:, np.newaxis, ...]

    y_out = np.asarray(
        y_out,
        dtype=np.int64,
    )
    uids_out = np.asarray(uids_out)
    aug_ids = np.asarray(
        aug_ids,
        dtype=np.int64,
    )

    return X, y_out, uids_out, aug_ids

In [7]:
# %%
train_augment_count = 5

X_train_val_aug, y_train_val_aug, uids_train_val_aug, aug_ids_train_val = (
    build_augmented_dataset(
        train_val_paths,
        train_val_y,
        train_val_uids,
        augment_count=train_augment_count,
        base_seed=1000,
        checkpoint_dir=(
            project_dir
            / "data"
            / "checkpoints"
            / "train_augmented"
        ),
    )
)
X_test, y_test_processed, uids_test_processed, aug_ids_test = (
    build_augmented_dataset(
        test_paths,
        test_y,
        test_uids,
        augment_count=1,
        base_seed=5000,
        checkpoint_dir=(
            project_dir
            / "data"
            / "checkpoints"
            / "test_processed"
        ),
    )
)


1/1089 | xjn6fokz | aug 1/5 | wczytano
1/1089 | xjn6fokz | aug 2/5 | wczytano
1/1089 | xjn6fokz | aug 3/5 | wczytano
1/1089 | xjn6fokz | aug 4/5 | wczytano
1/1089 | xjn6fokz | aug 5/5 | wczytano
2/1089 | po6j0m3l | aug 1/5 | wczytano
2/1089 | po6j0m3l | aug 2/5 | wczytano
2/1089 | po6j0m3l | aug 3/5 | wczytano
2/1089 | po6j0m3l | aug 4/5 | wczytano
2/1089 | po6j0m3l | aug 5/5 | wczytano
3/1089 | 45bg34iq | aug 1/5 | wczytano
3/1089 | 45bg34iq | aug 2/5 | wczytano
3/1089 | 45bg34iq | aug 3/5 | wczytano
3/1089 | 45bg34iq | aug 4/5 | wczytano
3/1089 | 45bg34iq | aug 5/5 | wczytano
4/1089 | g5rifwr0 | aug 1/5 | wczytano
4/1089 | g5rifwr0 | aug 2/5 | wczytano
4/1089 | g5rifwr0 | aug 3/5 | wczytano
4/1089 | g5rifwr0 | aug 4/5 | wczytano
4/1089 | g5rifwr0 | aug 5/5 | wczytano
5/1089 | 6e07fpy1 | aug 1/5 | wczytano
5/1089 | 6e07fpy1 | aug 2/5 | wczytano
5/1089 | 6e07fpy1 | aug 3/5 | wczytano
5/1089 | 6e07fpy1 | aug 4/5 | wczytano
5/1089 | 6e07fpy1 | aug 5/5 | wczytano
6/1089 | 9zxqerbk | aug 1

In [8]:
# %%
np.savez_compressed(
    project_dir / "data" / "processed_volumes_ants_aug_train_val.npz",
    X=X_train_val_aug.astype(np.float16),
    y=y_train_val_aug,
    uid=uids_train_val_aug,
    aug_id=aug_ids_train_val,
)

np.savez_compressed(
    project_dir / "data" / "processed_volumes_ants_aug_test.npz",
    X=X_test.astype(np.float16),
    y=y_test_processed,
    uid=uids_test_processed,
    aug_id=aug_ids_test,
)

In [5]:
# %%
train_val_data = np.load(
    project_dir / "data" / "processed_volumes_ants_aug_train_val.npz",
    allow_pickle=False,
)

test_data = np.load(
    project_dir / "data" / "processed_volumes_ants_aug_test.npz",
    allow_pickle=False,
)

X_train_val_aug = train_val_data["X"].astype(np.float32)
y_train_val_aug = train_val_data["y"].astype(np.int64)
uids_train_val_aug = train_val_data["uid"]
aug_ids_train_val = train_val_data["aug_id"]

X_test = test_data["X"].astype(np.float32)
y_test = test_data["y"].astype(np.int64)
uids_test = test_data["uid"]

print("X_train_val_aug:", X_train_val_aug.shape)
print("X_test:", X_test.shape)

X_train_val_aug: (5445, 1, 50, 40, 35)
X_test: (273, 1, 50, 40, 35)


In [6]:
# %%
class VolumeDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        volume = torch.from_numpy(self.X[index]).float()
        label = torch.tensor(self.y[index], dtype=torch.float32)

        return volume, label

In [7]:
# %%
class Simple3DCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv3d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.AdaptiveAvgPool3d(1),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)

        return x.squeeze(1)

In [12]:
# %%
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Urządzenie:", device)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

epochs = 2000
batch_size = 8
early_stopping_patience = 100
min_delta = 0.0001

unique_train_val_uids = train_val_uids
unique_train_val_y = train_val_y

fold_results = []
fold_model_paths = []

Urządzenie: cuda


In [13]:
# %%
from datetime import datetime

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")


for fold, (train_idx, val_idx) in enumerate(
    cv.split(unique_train_val_uids, unique_train_val_y),
    start=1,
):
    print(f"\n===== FOLD {fold} =====")

    fold_train_uids = unique_train_val_uids[train_idx]
    fold_val_uids = unique_train_val_uids[val_idx]

    train_mask = np.isin(uids_train_val_aug, fold_train_uids)

    val_mask = (
        np.isin(uids_train_val_aug, fold_val_uids)
        & (aug_ids_train_val == 0)
    )

    X_train = X_train_val_aug[train_mask]
    y_train = y_train_val_aug[train_mask]

    X_val = X_train_val_aug[val_mask]
    y_val = y_train_val_aug[val_mask]

    print("X_train:", X_train.shape, np.unique(y_train, return_counts=True))
    print("X_val:", X_val.shape, np.unique(y_val, return_counts=True))

    train_dataset = VolumeDataset(X_train, y_train)
    val_dataset = VolumeDataset(X_val, y_val)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
    )

    validation_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    model = Simple3DCNN().to(device)
    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=0.0001,
        weight_decay=0.0001,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.1,
        patience=15,
        min_lr=1e-5,
    )

    best_validation_loss = float("inf")
    best_model_state = None
    epochs_without_improvement = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for volumes_batch, labels_batch in train_loader:
            volumes_batch = volumes_batch.to(device)
            labels_batch = labels_batch.to(device)

            optimizer.zero_grad()

            logits = model(volumes_batch)
            loss = criterion(logits, labels_batch)

            loss.backward()
            optimizer.step()

            train_loss += loss.item() * len(labels_batch)

        train_loss /= len(train_dataset)

        model.eval()
        validation_loss = 0.0

        with torch.no_grad():
            for volumes_batch, labels_batch in validation_loader:
                volumes_batch = volumes_batch.to(device)
                labels_batch = labels_batch.to(device)

                logits = model(volumes_batch)
                loss = criterion(logits, labels_batch)

                validation_loss += loss.item() * len(labels_batch)

        validation_loss /= len(val_dataset)

        current_lr = optimizer.param_groups[0]["lr"]
        scheduler.step(validation_loss)
        new_lr = optimizer.param_groups[0]["lr"]

        if new_lr < current_lr:
            print(
                f"LR zmniejszony: {current_lr:.8f} -> {new_lr:.8f}",
                flush=True,
            )
            epochs_without_improvement = 0

        if validation_loss < best_validation_loss - min_delta:
            best_validation_loss = validation_loss
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        print(
            f"Fold {fold} | Epoka {epoch + 1}/{epochs} | "
            f"train: {train_loss:.4f} | "
            f"validation: {validation_loss:.4f} | "
            f"lr: {current_lr:.8f} | "
            f"no_improve: {epochs_without_improvement}",
            flush=True,
        )

        if epochs_without_improvement > early_stopping_patience:
            print(
                f"Early stopping: fold {fold}, epoka {epoch + 1}, "
                f"best validation loss: {best_validation_loss:.4f}"
            )
            break

    model.load_state_dict(best_model_state)

    model_path = outputs_dir / f"ants_aug_model{run_id}_fold_{fold}_val_{best_validation_loss}.pth"
    torch.save(model.state_dict(), model_path)

    fold_model_paths.append(model_path)
    fold_results.append(best_validation_loss)

    print(f"Fold {fold} najlepszy validation loss: {best_validation_loss:.4f}")
    print(f"Zapisano: {model_path}")

print("\nWyniki foldów:", fold_results)
print("Średni validation loss:", np.mean(fold_results))


===== FOLD 1 =====
X_train: (4355, 1, 50, 40, 35) (array([0, 1]), array([1965, 2390]))
X_val: (218, 1, 50, 40, 35) (array([0, 1]), array([ 99, 119]))
Fold 1 | Epoka 1/2000 | train: 0.6869 | validation: 0.6854 | lr: 0.00010000 | no_improve: 0
Fold 1 | Epoka 2/2000 | train: 0.6822 | validation: 0.6757 | lr: 0.00010000 | no_improve: 0
Fold 1 | Epoka 3/2000 | train: 0.6671 | validation: 0.6507 | lr: 0.00010000 | no_improve: 0
Fold 1 | Epoka 4/2000 | train: 0.6293 | validation: 0.5990 | lr: 0.00010000 | no_improve: 0
Fold 1 | Epoka 5/2000 | train: 0.5673 | validation: 0.5394 | lr: 0.00010000 | no_improve: 0
Fold 1 | Epoka 6/2000 | train: 0.5296 | validation: 0.5106 | lr: 0.00010000 | no_improve: 0
Fold 1 | Epoka 7/2000 | train: 0.5008 | validation: 0.4915 | lr: 0.00010000 | no_improve: 0
Fold 1 | Epoka 8/2000 | train: 0.4885 | validation: 0.4796 | lr: 0.00010000 | no_improve: 0
Fold 1 | Epoka 9/2000 | train: 0.4743 | validation: 0.4781 | lr: 0.00010000 | no_improve: 0
Fold 1 | Epoka 10/200

KeyboardInterrupt: 

In [14]:
from pathlib import Path

fold_model_paths = [
    Path.cwd() / f"ants_aug_model_fold_{fold}.pth"
    for fold in range(1, 6)
]

for model_path in fold_model_paths:
    if not model_path.exists():
        raise FileNotFoundError(
            f"Nie znaleziono: {model_path}"
        )

    print("Znaleziono:", model_path.name)

Znaleziono: ants_aug_model_fold_1.pth
Znaleziono: ants_aug_model_fold_2.pth
Znaleziono: ants_aug_model_fold_3.pth
Znaleziono: ants_aug_model_fold_4.pth
Znaleziono: ants_aug_model_fold_5.pth


In [16]:
# %%
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_dataset = VolumeDataset(X_test, y_test)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
)

all_fold_probs = []

for model_path in fold_model_paths:
    model = Simple3DCNN().to(device)
    model.load_state_dict(
        torch.load(model_path, map_location=device)
    )
    model.eval()

    fold_probs = []

    with torch.no_grad():
        for volumes_batch, labels_batch in test_loader:
            volumes_batch = volumes_batch.to(device)

            logits = model(volumes_batch)
            probs = torch.sigmoid(logits)

            fold_probs.extend(probs.cpu().numpy())

    all_fold_probs.append(np.asarray(fold_probs))

all_fold_probs = np.asarray(all_fold_probs)

test_probs = all_fold_probs.mean(axis=0)
test_probs = np.clip(test_probs, 0.001, 0.999)

test_pred_class = (test_probs >= 0.5).astype(int)

print("Liczba modeli:", len(fold_model_paths))
print("Shape all_fold_probs:", all_fold_probs.shape)
print("Min pred:", test_probs.min())
print("Max pred:", test_probs.max())

print("Log loss:", log_loss(y_test, test_probs))
print("AUROC:", roc_auc_score(y_test, test_probs))
print("Accuracy:", accuracy_score(y_test, test_pred_class))
print("F1:", f1_score(y_test, test_pred_class))

C:\Users\abram\AppData\Local\Temp\ipykernel_12380\3438090669.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(model_path, map_location=device)


Liczba modeli: 5
Shape all_fold_probs: (5, 273)
Min pred: 0.006224633
Max pred: 0.999
Log loss: 0.3001060485839844
AUROC: 0.9473712737127371
Accuracy: 0.8644688644688645
F1: 0.8810289389067524


In [17]:
train_dataset = VolumeDataset(
    X_train_val_aug,
    y_train_val_aug,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
)

all_fold_train_probs = []

for model_path in fold_model_paths:
    print("Wczytuję:", model_path.name)

    model = Simple3DCNN().to(device)

    state_dict = torch.load(
        model_path,
        map_location=device,
        weights_only=True,
    )

    model.load_state_dict(
        state_dict,
        strict=True,
    )

    model.eval()

    fold_train_probs = []

    with torch.inference_mode():
        for volumes_batch, _ in train_loader:
            volumes_batch = volumes_batch.to(device)

            logits = model(volumes_batch)
            probabilities = torch.sigmoid(logits)

            fold_train_probs.extend(
                probabilities.cpu().numpy()
            )

    all_fold_train_probs.append(
        np.asarray(
            fold_train_probs,
            dtype=np.float32,
        )
    )

Wczytuję: ants_aug_model_fold_1.pth
Wczytuję: ants_aug_model_fold_2.pth
Wczytuję: ants_aug_model_fold_3.pth
Wczytuję: ants_aug_model_fold_4.pth
Wczytuję: ants_aug_model_fold_5.pth


In [19]:
all_fold_train_probs = np.asarray(
    all_fold_train_probs,
    dtype=np.float32,
)

print(
    "Shape:",
    all_fold_train_probs.shape,
)

assert all_fold_train_probs.shape == (
    5,
    len(y_train_val_aug),
)

meta_train_antspyx = pd.DataFrame({
    "uid": np.asarray(uids_train_val_aug),
    "y_true": np.asarray(y_train_val_aug),
})

for fold_index in range(5):
    meta_train_antspyx[
        f"p_antspyx_fold_{fold_index + 1}"
    ] = np.clip(
        all_fold_train_probs[fold_index],
        0.001,
        0.999,
    )

meta_train_antspyx_path = (
    Path.cwd()
    / "meta_train_predictions_antspyx.csv"
)

meta_train_antspyx.to_csv(
    meta_train_antspyx_path,
    index=False,
)

print("Zapisano:", meta_train_antspyx_path)
print("Shape:", meta_train_antspyx.shape)

display(meta_train_antspyx.head())

Shape: (5, 5445)
Zapisano: C:\Users\abram\PycharmProjects\DrivenData_DAT_Parkinsons\src\notebooks\EDA5\meta_train_predictions_antspyx.csv
Shape: (5445, 7)


,uid,y_true,p_antspyx_fold_1,p_antspyx_fold_2,p_antspyx_fold_3,p_antspyx_fold_4,p_antspyx_fold_5
0,xjn6fokz,0,0.368493,0.179227,0.440302,0.166970,0.294193
1,xjn6fokz,0,0.248782,0.128393,0.316252,0.174138,0.344032
2,xjn6fokz,0,0.308822,0.143640,0.381715,0.215920,0.204510
3,xjn6fokz,0,0.287798,0.112141,0.324932,0.197842,0.296567
4,xjn6fokz,0,0.333308,0.112506,0.317143,0.185755,0.286414
